# piControl: annual anomalies of every variable (data preparation)

Step 1 of 2 of the data preparation, and the only notebook that reads the raw CESM2 output. One file per variable in `anom_dir`, named `<name>_annual_anom.nc`, plus the annual PDV and AMV indices.

Every file holds **annual means, the first 100 years dropped (spin-up), linear trend removed, anomalies** (`clip_detrend_anom`), 1100 years (101–1200). Filtering (decade means, Lanczos low-pass) is not done here; each notebook applies the one it needs.

| file | built in | raw input |
|---|---|---|
| `SST_annual_anom.nc` | section 1 | `ts`, ocean only |
| `PDV_AMV_annual.nc` | section 1 | from the SST anomalies |
| `SWTOA_annual_anom.nc`, `LWTOA_annual_anom.nc` | section 2 | `rsdt`, `rsut`, `rlut` |
| `LTS_annual_anom.nc` | section 3 | `ta` (700 hPa), `tas`, `ps` from `cl` |
| `CLDLOW/CLDMED/CLDHGH_annual_anom.nc` | section 4 | `cl` |

Raw input: one folder per CMIP variable in `data_dir` (e.g. `data_dir/ts/*.nc`); all files of a folder are read together, so they do not have to be merged first. SST also needs the land fraction `sftlf` (`mask_path`).

A file that is already there is kept (`overwrite = False`), so a run only builds what is missing.

Used by: `02_piControl_decadal_storylines.ipynb` (step 2: decades and storylines), `03_piControl_lowpass_MLR.ipynb`, `04_piControl_state_transition.ipynb`.

In [ ]:
import pathlib

import xarray as xr

from functions import (
    annual_anomalies_picontrol,
    clip_detrend_anom,
    ocean_mask,
    compute_pdv_index,
    compute_amv_index,
    lower_tropospheric_stability,
)

## Config

In [ ]:
# raw CESM2 piControl output, one folder per CMIP variable
data_dir = pathlib.Path('/climca/data/CESM2_LE_mon/piControl')
mask_path = '/climca/data/CESM2_LE_mon/mask/sftlf_fx_CESM2_piControl_r1i1p1f1_gn.nc'

# all annual anomalies live here, one file per variable: <name>_annual_anom.nc
anom_dir = pathlib.Path('/path/to/data')
index_file = anom_dir / 'PDV_AMV_annual.nc'

# False: a file that is already there is kept; True: built again and overwritten
overwrite = False

# cloud levels (section 4): pressure bands in Pa (CAM native) and overlap of the layers
# "max" = CAM native; "random" = ISCCP style (Luo et al. 2023: splits at 68000 / 44000 Pa)
cloud_bounds = {
    'CLDLOW': (70000.0, 120000.0),   # 700-1200 hPa
    'CLDMED': (40000.0, 70000.0),    # 400-700 hPa
    'CLDHGH': (5000.0, 40000.0),     # 50-400 hPa
}
cloud_overlap = 'max'

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
encoding = {'dtype': 'float32', 'zlib': True, 'complevel': 4}

## Helpers

In [3]:
def anom_path(name):
    return anom_dir / f'{name}_annual_anom.nc'


def needed(path):
    """True if ``path`` has to be built (missing, or overwrite is set)."""
    return overwrite or not path.exists()


def open_var(name, chunks=None, **kwargs):
    """Monthly raw output of one CMIP variable, all files of its folder."""
    paths = sorted((data_dir / name).glob('*.nc'))
    if not paths:
        raise FileNotFoundError(f'no .nc files in {data_dir / name}')
    return xr.open_mfdataset(
        paths, combine='by_coords', decode_times=time_coder,
        chunks=chunks or {'time': 240, 'lat': 96, 'lon': 96}, **kwargs,
    )


def write_anom(da, name, **attrs):
    """Write the annual anomalies of ``name``; ``attrs`` say what the field is."""
    da = da.rename(name).astype('float32')
    da.attrs = {
        **attrs,
        'anomalies': 'annual means, first 100 years dropped, linear trend removed',
        'source': 'CESM2 piControl, 01_piControl_annual_anomalies.ipynb',
    }
    path = anom_path(name)
    da.to_netcdf(path, encoding={name: encoding})
    print(f'wrote {path}')


def read_anom(name):
    return xr.open_dataarray(anom_path(name), decode_times=time_coder)

## What is already computed

In [ ]:
expected = {
    'SST': 'section 1',
    'SWTOA': 'section 2',
    'LWTOA': 'section 2',
    'LTS': 'section 3',
    'CLDLOW': 'section 4',
    'CLDMED': 'section 4',
    'CLDHGH': 'section 4',
}
for name, origin in expected.items():
    path = anom_path(name)
    if path.exists():
        da = read_anom(name)
        years = da['time'].dt.year.values
        print(f"{name:7s} {years[0]}-{years[-1]} ({da.sizes['time']} years)  {path.name}")
    else:
        print(f'{name:7s} missing -> {origin}')
print(f"indices {'there' if index_file.exists() else 'missing -> section 1'}: {index_file.name}")

## 1 SST and the PDV/AMV indices

SST = `ts` over the ocean (`sftlf` < 50 %). The slow part is reading `ts` once.

The indices are built from the SST anomalies. Area means and the linear detrending commute, so this gives the same series as building the index from the raw SST and detrending it afterwards. They are stored unfiltered.

In [5]:
if needed(anom_path('SST')):
    ocean = ocean_mask(mask_path)
    sst = annual_anomalies_picontrol(open_var('ts')['ts'], mask=ocean).compute()
    write_anom(sst, 'SST', units='K',
               long_name='sea surface temperature anomaly (ts over the ocean, sftlf < 50 %)')
    del sst
else:
    print('SST: kept')

/home/lbraun/.conda/envs/maenv/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'sftlf' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/lbraun/.conda/envs/maenv/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'ts' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/tmp/ipykernel_862178/3329747057.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "lat" starting at index 96. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_mfdataset(
/tmp/ipykernel_862178/3329747057.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 96. This could degrade performance. Instead, consider rechunking after loading.
  ret

wrote /home/lbraun/data/SST_annual_anom.nc


In [7]:
if needed(index_file):
    sst = read_anom('SST').astype('float64').load()
    indices = xr.Dataset({'PDV': compute_pdv_index(sst), 'AMV': compute_amv_index(sst)})
    indices['PDV'].attrs = {
        'units': 'K',
        'long_name': 'PDV index (Pacific tripole), annual anomaly',
        'definition': '10S-10N 170-270E minus half the sum of 25-45N 140-215E and 50-15S 150-200E',
    }
    indices['AMV'].attrs = {
        'units': 'K',
        'long_name': 'AMV index, annual anomaly',
        'definition': '0-60N 280-360E minus 60S-60N, area weighted',
    }
    indices.attrs = {
        'source': 'CESM2 piControl ts over the ocean, 01_piControl_annual_anomalies.ipynb',
        'anomalies': 'annual means, first 100 years dropped, linear trend removed; not filtered',
    }
    indices.to_netcdf(index_file)
    print(f'wrote {index_file}')
else:
    print('indices: kept')

wrote /home/lbraun/data/PDV_AMV_annual.nc


## 2 TOA shortwave and longwave

Positive downward: SW = `rsdt − rsut` (absorbed solar), LW = `−rlut`. The net flux is not stored, it is SW + LW. `rsdt` is the same in every year of piControl, so SW anomalies equal those of `−rsut`.

In [9]:
if needed(anom_path('SWTOA')):
    sw = annual_anomalies_picontrol(open_var('rsdt')['rsdt'] - open_var('rsut')['rsut']).compute()
    write_anom(sw, 'SWTOA', units='W m-2', positive='down',
               long_name='net downward shortwave at TOA (rsdt - rsut), anomaly')
    del sw
else:
    print('SWTOA: kept')

if needed(anom_path('LWTOA')):
    lw = annual_anomalies_picontrol(-open_var('rlut')['rlut']).compute()
    write_anom(lw, 'LWTOA', units='W m-2', positive='down',
               long_name='net downward longwave at TOA (-rlut), anomaly')
    del lw
else:
    print('LWTOA: kept')

SWTOA: kept
LWTOA: kept


## 3 Lower tropospheric stability

LTS = θ(700 hPa) − θ(surface) with θ(surface) = tas·(p0/ps)^κ and the surface pressure from the `cl` files (`functions.lower_tropospheric_stability`). 

In [10]:
if needed(anom_path('LTS')):
    # plev has to be chunked: only 700 hPa of the levels is used
    ta = open_var('ta', chunks={'time': 240, 'plev': 1, 'lat': 96, 'lon': 96})['ta']
    tas = open_var('tas')['tas']
    # only ps is kept from the cl files, so the 3D cloud field is never touched
    ps = open_var('cl', preprocess=lambda ds: ds[['ps']])['ps']
    ta, tas, ps = xr.align(ta, tas, ps, join='exact')     # same months, same grid

    lts = lower_tropospheric_stability(ta, tas, ps)
    complete = lts.notnull().resample(time='YS').sum() == 12
    annual = lts.resample(time='YS').mean().where(complete).compute()
    write_anom(clip_detrend_anom(annual), 'LTS', units='K',
               long_name='lower tropospheric stability theta(700 hPa) - theta(surface), anomaly')
    del ta, tas, ps, lts, annual
else:
    print('LTS: kept')

LTS: kept


## 4 Clouds

`CLDLOW`, `CLDMED` and `CLDHGH` from the 3D cloud fraction `cl` on the hybrid levels:

1. pressure of every level: p = a·p0 + b·ps (Pa),
2. each band (`cloud_bounds`) collapsed to one value per column: maximum over the levels inside the band (`cloud_overlap = 'max'`, CAM native) or random overlap 1 − ∏(1 − cl),
3. annual means (the 12 months weighted equally), then `clip_detrend_anom`.

Banding on p, so the order of `lev` (top to surface) does not matter. Output in % (0–100), as CAM writes cloud cover.

In [ ]:
cloud_names = list(cloud_bounds)
if any(needed(anom_path(name)) for name in cloud_names):
    ds = open_var('cl')
    cl = ds['cl']
    if cl.attrs.get('units', '').strip() in ('%', 'percent'):    # % -> fraction for the overlap
        cl = cl / 100.0
    a_p0 = ds['ap'] if 'ap' in ds else ds['a'] * ds['p0']         # CESM2 files: a, b, p0
    p = (a_p0 + ds['b'] * ds['ps']).broadcast_like(cl)

    def collapse(bounds):
        top, bottom = bounds
        inside = (p > top) & (p <= bottom)
        if cloud_overlap == 'max':
            return cl.where(inside).max('lev')
        if cloud_overlap == 'random':
            return 1.0 - (1.0 - cl.where(inside, 0.0)).prod('lev')
        raise ValueError(cloud_overlap)

    clouds = xr.Dataset({name: collapse(bounds) * 100 for name, bounds in cloud_bounds.items()})
    annual = clouds.resample(time='YS').mean().compute()           # the slow step: reads cl once
    for name, (top, bottom) in cloud_bounds.items():
        if needed(anom_path(name)):
            write_anom(clip_detrend_anom(annual[name]), name, units='%',
                       long_name=f'{name} cloud cover ({cloud_overlap} overlap, '
                                 f'{bottom / 100:.0f}-{top / 100:.0f} hPa), anomaly')
        else:
            print(f'{name}: kept')
    del ds, cl, p, clouds, annual
else:
    print('clouds: kept')